# AgentSim 自定义功能测试

这个 notebook 不加载任何 benchmark，只验证如何从外部创建自定义任务、环境状态和工具，并通过标准 EnvSim API 执行。


In [ ]:
%pip install -q --no-deps --no-build-isolation -e .


In [ ]:
from dataclasses import dataclass, field
from agentsim import (
    EnvSim, EnvState, TaskDefine, ScenarioDefine,
    ToolCall, ToolDefine, ToolExecutor, ToolRiskLevel,
)

print("AgentSim:", __import__("agentsim").__version__)


## 1. 定义自定义环境状态


In [ ]:
@dataclass
class CounterState(EnvState):
    value: int = 0
    history: list[str] = field(default_factory=list)


## 2. 定义并注册环境工具


In [ ]:
def increment(state: CounterState, amount: int = 1):
    state.value += amount
    state.history.append(f"increment({amount})")
    return {"value": state.value}

executor = ToolExecutor()
executor.register_tool(
    "increment",
    increment,
    ToolDefine(
        name="increment",
        description="Increase the counter.",
        parameters={"type": "object", "properties": {"amount": {"type": "integer"}}},
        risk_level=ToolRiskLevel.SAFE,
        category="write",
    ),
)


## 3. 创建任务和环境


In [ ]:
scenario = ScenarioDefine(
    scenario_id="custom_counter",
    name="Custom Counter",
    description="A tiny user-defined environment.",
    state_schema=["value", "history"],
    tools={"counter": ["increment"]},
)

def init_state(initial_state, seed):
    return CounterState(value=initial_state.get("value", seed), history=[])

class CounterEnv(EnvSim):
    pass

env = CounterEnv(scenario=scenario, init_state=init_state, executor=executor)
task = TaskDefine(
    task_id="counter_task",
    scenario="custom_counter",
    description="Increase the counter by three.",
    initial_state={"value": 10},
    max_steps=5,
)
observation = env.reset(task, instruction=task.description)
print("Initial:", observation)


## 4. 执行工具并查看状态与 Trace


In [ ]:
result = env.call_tool(
    "increment",
    {"amount": 3},
    agent_id="custom-agent",
)
print("Tool result:", result.to_dict())
print("State:", env.env_state.to_dict())
print("Trace events:", len(env.get_trace().events))


## 5. 自定义任务可以直接重复运行


In [ ]:
second_task = TaskDefine(
    task_id="counter_task_2",
    scenario="custom_counter",
    description="Increase the counter by two.",
    initial_state={"value": 20},
)
env.reset(second_task, instruction=second_task.description)
env.call_tool("increment", {"amount": 2})
assert env.env_state.value == 22
print("Custom task passed:", second_task.task_id)
